# Stage 2 – Method 1 Simplified Margin
Batch-level approximation of margin loss.

In [ ]:

import os
import sys
from pathlib import Path

use_colab = 'google.colab' in sys.modules
if use_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    base_dir = Path('/content/drive/MyDrive/liveness_detection_vae')
else:
    base_dir = Path.cwd()

project_dir = base_dir
split_json = project_dir / 'data_split.json'
base_data_dir = project_dir
pretrained_path = project_dir / 'runs' / 'stage1_pretrain_colab' / 'stage1_pretrained.pt'
save_dir = project_dir / 'runs' / 'stage2_method1_margin_simple_colab'
save_dir.mkdir(parents=True, exist_ok=True)

os.chdir(project_dir)
print('Project dir:', project_dir)
print('Split JSON:', split_json)
print('Pretrained:', pretrained_path)
print('Save dir:', save_dir)


In [ ]:

import time
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from config_bandvae import get_config
from dataset_stage2 import Stage2Dataset
from model_bandvae import BandSplitVAE, band_split_vae_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = get_config('full')
config.T_fixed = 300
config.fc_low = 2.0
config.fc_high = 8.0
config.filter_order = 4
config.C_h = 48
config.C_z = 12
config.dilations = [1, 2, 4]
config.lr = 1e-4
config.epochs = 10
config.batch_size = 64
config.num_workers = 4
config.device = device
print(config)


In [ ]:

train_dataset = Stage2Dataset(
    split_json_path=str(split_json),
    split_name='stage2_train',
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=True,
    base_dir=str(base_data_dir),
)

val_dataset = Stage2Dataset(
    split_json_path=str(split_json),
    split_name='final_test',
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=False,
    base_dir=str(base_data_dir),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print('Train samples:', len(train_dataset), 'Val samples:', len(val_dataset))


In [ ]:

checkpoint = torch.load(pretrained_path, map_location=config.device)

model = BandSplitVAE(
    C_in_per_band=config.C_in_per_band,
    C_h=config.C_h,
    C_z=config.C_z,
    dilations=config.dilations,
).to(config.device)
model.load_state_dict(checkpoint['model_state_dict'])

optimizer = optim.Adam(model.parameters(), lr=config.lr)
use_amp = config.device == 'cuda'


In [ ]:

MARGIN = 0.5
LAMBDA_MARGIN = 1.0

def validate(model, loader, device, margin):
    model.eval()
    real_losses, fake_losses = [], []

    with torch.no_grad():
        for x_lf, x_bp, x_hf, labels in loader:
            x_lf = x_lf.to(device)
            x_bp = x_bp.to(device)
            x_hf = x_hf.to(device)
            labels = labels.to(device)

            recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}

            loss, loss_dict = band_split_vae_loss(
                recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
            )

            per_sample = loss / labels.size(0)
            for lbl in labels:
                if lbl == 0:
                    real_losses.append(per_sample.item())
                else:
                    fake_losses.append(per_sample.item())

    real_mean = sum(real_losses) / len(real_losses) if real_losses else 0.0
    fake_mean = sum(fake_losses) / len(fake_losses) if fake_losses else 0.0
    fake_penalty = max(0.0, margin - fake_mean)
    total = real_mean + LAMBDA_MARGIN * fake_penalty
    return {
        'total': total,
        'real_rec': real_mean,
        'fake_rec': fake_mean,
        'separation': fake_mean - real_mean,
    }


def train_epoch(model, loader, optimizer, device, margin, use_amp=True):
    model.train()
    total_loss = 0.0
    real_losses, fake_losses = [], []
    n_batches = len(loader)
    scaler = torch.amp.GradScaler('cuda') if use_amp and torch.cuda.is_available() else None

    for x_lf, x_bp, x_hf, labels in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()

        if scaler is not None:
            with torch.amp.autocast('cuda'):
                recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
                targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
                betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
                loss, loss_dict = band_split_vae_loss(
                    recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
                )
                batch_size = labels.size(0)
                avg_per_sample = loss / batch_size
                n_real = (labels == 0).sum().item()
                n_fake = (labels == 1).sum().item()
                fake_penalty = torch.clamp(margin - avg_per_sample, min=0.0)
                total_batch_loss = avg_per_sample * n_real + LAMBDA_MARGIN * fake_penalty * n_fake
            scaler.scale(total_batch_loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
            loss, loss_dict = band_split_vae_loss(
                recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
            )
            batch_size = labels.size(0)
            avg_per_sample = loss / batch_size
            n_real = (labels == 0).sum().item()
            n_fake = (labels == 1).sum().item()
            fake_penalty = torch.clamp(margin - avg_per_sample, min=0.0)
            total_batch_loss = avg_per_sample * n_real + LAMBDA_MARGIN * fake_penalty * n_fake
            total_batch_loss.backward()
            optimizer.step()

        total_loss += total_batch_loss.item()
        per_sample = loss_dict['total'] / labels.size(0)
        for lbl in labels:
            if lbl == 0:
                real_losses.append(per_sample)
            else:
                fake_losses.append(per_sample)

    real_mean = sum(real_losses) / len(real_losses) if real_losses else 0.0
    fake_mean = sum(fake_losses) / len(fake_losses) if fake_losses else 0.0
    return {
        'total': total_loss / n_batches,
        'real_rec': real_mean,
        'fake_rec': fake_mean,
        'separation': fake_mean - real_mean,
    }


In [ ]:

best_sep = -float('inf')
best_path = save_dir / 'stage2_method1_margin_simple_best.pt'

for epoch in range(1, config.epochs + 1):
    start = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, config.device, MARGIN, use_amp=use_amp)
    val_loss = validate(model, val_loader, config.device, MARGIN)
    duration = (time.time() - start) / 60

    print(
        f"Epoch {epoch}/{config.epochs} | "
        f"train total {train_loss['total']:.4f} real {train_loss['real_rec']:.4f} fake {train_loss['fake_rec']:.4f} sep {train_loss['separation']:.4f} | "
        f"val total {val_loss['total']:.4f} real {val_loss['real_rec']:.4f} fake {val_loss['fake_rec']:.4f} sep {val_loss['separation']:.4f} | "
        f"{duration:.1f} min"
    )

    if val_loss['separation'] > best_sep:
        best_sep = val_loss['separation']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_loss,
            'config': config,
            'margin': MARGIN,
            'lambda_margin': LAMBDA_MARGIN,
        }, best_path)
        print(f'Saved best to {best_path}')
